# 59 · Decoupled classifier (cRT) — huấn luyện lại **riêng** `GroupFC` với lấy mẫu cân bằng

## Lý lẽ (hướng #2, `PLAN_STAGE5`)

Nhãn hiếm không thiếu *biểu diễn* — backbone nhìn `Lobar Atelectasis` bằng đúng đôi mắt nhìn mọi
nhãn khác. Nó thiếu **số lần cập nhật classifier**: 129 ca dương / 258,871 ảnh ⇒ `GroupFC` của nhãn
đó nhận ~129 gradient dương mỗi epoch, so với hàng chục nghìn của nhãn head. Kang et al.
(decoupling): giữ biểu diễn học với phân bố **tự nhiên** (đã có), huấn luyện lại **riêng
classifier** với lấy mẫu cân bằng lớp.

## Cắt ở đâu — và vì sao gần như miễn phí

```
backbone (dong cung)
  -> Tier1Img -> pin -> Tier1(EHR)          BIEU DIEN     } dong cung, CACHE dau ra
  -> XAttn(Q=40 query lop, K/V=patch)                     }   H [N, 40, 512]
  -> GroupFC                                CLASSIFIER    <- CHI mo cai nay
```

Cache `H` một lần (fp16, ~15 GB/seed trên đĩa local) rồi việc huấn luyện lại chỉ là **một lớp tuyến
tính trên đặc trưng đã cache** — vài giây mỗi epoch. Không mở `XAttn`: mở nó thì phải cache `kv`
1369×512 ⇒ 540 GB, không khả thi; đó là bậc leo sau, chỉ khi bậc này dương.

## Ba arm — arm giữa là thứ quyết định

```
base       GroupFC cua checkpoint, khong dong gi          moc (= tai lap ep10)
cRT_inst   huan luyen lai GroupFC, lay mau TU NHIEN       DOI CHUNG
cRT_bal    huan luyen lai GroupFC, lay mau CAN BANG       arm that

G_bal = cRT_bal - cRT_inst    <- CAU HOI CHINH, ghep cap hoan hao
cRT_inst - base               phan CHI do "huan luyen them" — thieu no thi G tron hai thu,
                              dung loi da thoi phong G_ehr o 52b va lam 56 khong doc duoc
```

## Giao thức đo — khoá trước (`58` quyết định)

Mốc và arm đều xử lý **y hệt**: ensemble **tất cả** epoch retrain × mọi seed có (cửa sổ `all` thắng
trên lát chọn ở cả hai seed trong `58`). So sánh bằng bootstrap **ghép cặp** theo **bệnh nhân**.

```
NHAN      G_bal tail > +0.005  VA  CI loai 0
KHONG     macro tut qua -0.002 (lay mau can bang co tac dung phu keo head xuong)
VUNG XAM  CI chua 0 nhung diem uoc luong > +0.003  ->  THEM SEED, khong ket luan
```

CI nửa-rộng của hiệu ghép cặp ở tail đo được `≈ ±0.005` — **sát ngưỡng**, chuẩn bị sẵn tinh thần
cần seed thứ ba.

## Chi phí

```
cache H     ~55 ph/seed x 2 seed (mot lan; suy luan toan bo 377k anh + phim truoc)
retrain     ~1 ph/arm/seed
tong        ~2 h, gan het la cache
```

## C0 · Config

In [4]:
import os, json, math, time, tarfile, io as _io, re, datetime
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = os.environ.get('CXRLT_DATA', '')
    assert DATA_DIR, 'khong phai Colab va chua dat CXRLT_DATA -> gan thang duong dan `planB_out`'
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024'); EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RUN_TAG = 'r1'
RES59 = os.path.join(OUT_DIR, 'stage4_nb59', RUN_TAG)
SRC53 = os.path.join(OUT_DIR, 'stage4_nb53', 'r3')       # nguon checkpoint + du doan tham chieu
ARM53, EP_REF = 't1_prior_both', 10
P_OOF = os.path.join(OUT_DIR, 'stage4_nb52a', 'xgb_oof.npy')
P_TOUCH = os.path.join(OUT_DIR, '_test_touches_nb59.jsonl')
def ensure_res():
    '''Drive FUSE danh roi thu muc RONG — goi TRUOC moi cum ghi.'''
    os.makedirs(RES59, exist_ok=True); return RES59
ensure_res()
assert os.path.isdir(OUT_DIR) and os.listdir(OUT_DIR), f'{OUT_DIR} khong doc duoc -> mount hong'
json.dump({'run_tag': RUN_TAG, 'notebook': '59'}, open(os.path.join(RES59, '_run_info.json'), 'w'))
LOCAL = '/content/cxrlt_518'
READY_X = os.path.join(OUT_DIR, 'cxrlt_518_ready')       # kho 41c C3-bis tao
CACHE_DIR = '/content'                                    # H fp16 ~15 GB/seed — dia LOCAL

SEEDS  = (42, 43)
SEED, SIZE = 42, 518
BATCH, ACCUM = 32, 8                                      # chi dung BATCH cho cache/suy luan
E_RT, LR_RT, WD_RT, BS_RT = 10, 1e-3, 1e-2, 4096          # retrain classifier
NW = min(8, (os.cpu_count() or 2))
D_MLD, D_ETOK, EHR_CLIP = 512, 256, 10.0
DO_TEST, N_TEST_EP = True, 3                              # N_TEST_EP khong dung — giu chu ky C0 cu

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'thiet bi: {torch.cuda.get_device_name(0)} | '
      f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB | AMP {AMP}')

P_MICRO = os.path.join(OUT_DIR, 'cxrlt_micro_full.parquet')
P_VOCAB = os.path.join(OUT_DIR, 'cxrlt_micro_vocab.json')
# 53 luu `{sk}_resume.pt` MOI epoch; sau epoch cuoi thi file resume CHINH LA trang thai ep10.
# Ban cuoi `{sk}.pt` chi ton tai neu vong seed chay tron ven — do de quy ca hai ten, uu tien ban
# cuoi, va in ro dang dung file nao thay vi chon cung mot ten roi chet nhu ban dau.
import glob as _glob
def find_ckpt(seed):
    for pat in (f's{seed}.pt', f's{seed}_resume.pt'):
        hit = (_glob.glob(os.path.join(SRC53, 'ckpt', pat))
               or _glob.glob(os.path.join(SRC53, '**', pat), recursive=True))
        if hit: return hit[0]
    return None
# LOC seed theo thu THAT SU co du hien vat, thay vi doi du ca `SEEDS`. Drive da nuot mat toan bo
# hien vat cua mot seed o r3 (cung lop su co da nuot trong so 41c; cac ban `... (1).pt` trong
# thu muc ckpt la dau hieu Drive dang xung dot ten). `G_bal` la phep so GHEP CAP TRONG cung mot
# seed, nen mot seed van cho phep do hop le — seed thu hai chi lam hep CI them chut.
CKPT_OF, _skip = {}, []
for s in SEEDS:
    c = find_ckpt(s)
    ok_npy = all(os.path.exists(os.path.join(SRC53, f'{pre}_{ARM53}_s{s}_ep{EP_REF}.npy'))
                 for pre in ('p', 'psl'))
    if c and ok_npy: CKPT_OF[s] = c
    else: _skip.append((s, 'thieu ckpt' if not c else 'thieu p_/psl_ ep%d' % EP_REF))
if _skip:
    print('BO QUA seed thieu hien vat: ' + ', '.join(f's{s} ({r})' for s, r in _skip))
    print(f'cac file .pt dang co duoi {SRC53}:')
    for f in sorted(_glob.glob(os.path.join(SRC53, '**', '*.pt'), recursive=True)):
        print(f'   {os.path.getsize(f)/1e6:8.1f} MB  ' + f.replace(SRC53, '...'))
assert CKPT_OF, (f'KHONG seed nao du hien vat duoi {SRC53} (can ckpt + p_/psl_ ep{EP_REF}). '
                 'Xem danh sach vua in, hoac doi SRC53 sang RUN_TAG khac.')
SEEDS = tuple(sorted(CKPT_OF))
for s in SEEDS: print(f'  ckpt s{s}: ' + CKPT_OF[s].replace(SRC53, '...'))
print(f'  -> chay voi {len(SEEDS)} seed: {SEEDS}'
      + ('   (mot seed: CI se rong hon, G_bal van ghep cap hop le)' if len(SEEDS) == 1 else ''))
for p, hd in ((P_MICRO, '29c'), (P_VOCAB, '29c'), (P_OOF, '52a')):
    assert os.path.exists(p), f'khong thay {p} -> chay {hd} truoc'

INV = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}
ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
assert (Y.dicom_id.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')
SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'; TE = ~(TR | VA)
_rs = np.random.default_rng(SEED); _su = np.unique(SUB[TR])
_sel = _rs.choice(_su, size=int(round(0.15*len(_su))), replace=False)
SL = TR & np.isin(SUB, _sel); TF_ = TR & ~SL              # CUNG cach cat nhu 35..57
assert not (set(SUB[SL]) & set(SUB[TF_])) and not (set(SUB[TR]) & set(SUB[VA]))
I_TF, I_SL, I_VA, I_TE = (np.where(m)[0] for m in (TF_, SL, VA, TE))
print(f'{len(ORDER):,} anh | khop {len(I_TF):,} | chon {len(I_SL):,} | val {len(I_VA):,} | '
      f'test {len(I_TE):,}')
print(f'nguon: {SRC53} arm `{ARM53}` ep{EP_REF} | retrain E={E_RT} LR={LR_RT} BS={BS_RT}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
thiet bi: NVIDIA RTX PRO 6000 Blackwell Server Edition | VRAM 102 GB | AMP torch.bfloat16
BO QUA seed thieu hien vat: s42 (thieu ckpt)
cac file .pt dang co duoi /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/stage4_nb53/r3:
      183.8 MB  .../ckpt/s43.pt
      550.4 MB  .../ckpt/s43_resume (1).pt
      550.4 MB  .../ckpt/s43_resume (2).pt
      550.4 MB  .../ckpt/s43_resume (3).pt
      550.4 MB  .../ckpt/s43_resume.pt
  ckpt s43: .../ckpt/s43.pt
  -> chay voi 1 seed: (43,)   (mot seed: CI se rong hon, G_bal van ghep cap hop le)
377,110 anh | khop 219,785 | chon 39,086 | val 39,293 | test 78,946
nguon: /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/stage4_nb53/r3 arm `t1_prior_both` ep10 | retrain E=10 LR=0.001 BS=4096


## C1 · EHR + túi vi khuẩn + XGB + phim trước — verbatim `53` r3

In [5]:
SPEC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
E_DF = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_ehr.parquet'))
_sid = ORDER.study_id.to_numpy()
COLS = [c for c in SPEC['Q_FULL'] if c in E_DF.columns]
_m = np.nan_to_num((E_DF.set_index('study_id')[COLS].reindex(_sid)
                    .apply(pd.to_numeric, errors='coerce').to_numpy('float32')), nan=0.0)
_b = np.array([np.isin(np.unique(_m[:, j]), (0.0, 1.0)).all() for j in range(_m.shape[1])])
_mu = _m[TF_].mean(0, keepdims=True); _sd = _m[TF_].std(0, keepdims=True) + 1e-6
_mu[:, _b] = 0.0; _sd[:, _b] = 1.0
E_ALL = np.clip((_m - _mu) / _sd, -EHR_CLIP, EHR_CLIP).astype('float32')
_keep = [i for i, c in enumerate(COLS) if not re.match(r'^micro_org_', c)]
assert len(COLS) - len(_keep) == 30, f'bo {len(COLS)-len(_keep)} cot micro_org_*, phai la 30'
COLS = [COLS[i] for i in _keep]; EHR_COL = E_ALL[:, _keep]

_MIC = pd.read_parquet(P_MICRO).set_index('study_id')
_VOC = json.load(open(P_VOCAB)); V_MIC = int(_VOC['V']) + 1; PAD_ID = V_MIC - 1
_mic = _MIC.reindex(_sid)
_ids = _mic.micro_org_ids.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
_cnt = _mic.micro_org_cnt.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
MIC_L = max(1, int(_ids.map(len).max())); assert MIC_L <= 64
MIDS = np.full((len(ORDER), MIC_L), PAD_ID, 'int64'); MCNT = np.zeros((len(ORDER), MIC_L), 'float32')
for r, (a, b) in enumerate(zip(_ids, _cnt)):
    if a: MIDS[r, :len(a)] = a; MCNT[r, :len(a)] = b

XGF = np.load(P_OOF).astype('float32')
assert XGF.shape == (len(ORDER), C) and np.isfinite(XGF).all()
_l = np.log(np.clip(XGF, 1e-6, 1-1e-6) / (1 - np.clip(XGF, 1e-6, 1-1e-6)))
XGB_F = np.clip((_l - _l[TF_].mean(0, keepdims=True)) / (_l[TF_].std(0, keepdims=True) + 1e-6),
                -EHR_CLIP, EHR_CLIP).astype('float32')
FEAT = np.concatenate([EHR_COL, XGB_F], 1)
print(f'FEAT {FEAT.shape} = {EHR_COL.shape[1]} EHR + {C} XGB | tui vi khuan V={V_MIC-1} dai {MIC_L}')

_BASE = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_base.parquet'))
_BASE['index_time'] = pd.to_datetime(_BASE.index_time)
_B = (_BASE[_BASE.study_id.isin(set(ORDER.study_id.unique()))]
      .sort_values(['subject_id','index_time']).reset_index(drop=True))
_B['prev_study'] = _B.groupby('subject_id', sort=False).study_id.shift(1)
_PV = dict(zip(_B.study_id.to_numpy(), _B.prev_study.to_numpy()))
_rw = {}
for _i, _s in enumerate(_sid): _rw.setdefault(_s, []).append(_i)
PREV_ROW = np.full(len(ORDER), -1, 'int64')
for _i, _s in enumerate(_sid):
    _p2 = _PV.get(_s, np.nan)
    if not (isinstance(_p2, float) and np.isnan(_p2)): PREV_ROW[_i] = _rw[int(_p2)][0]
_hp = PREV_ROW >= 0
print(f'anh co phim truoc: {_hp.sum():,} / {len(ORDER):,} ({_hp.mean():.1%})  (44a do 67.2%)')
assert _hp.mean() > 0.5 and (PREV_ROW[_hp] != np.arange(len(ORDER))[_hp]).all()

FEAT (377110, 257) = 217 EHR + 40 XGB | tui vi khuan V=305 dai 10
anh co phim truoc: 253,283 / 377,110 (67.2%)  (44a do 67.2%)


## C2 · Ảnh + backbone `ft` + loader — hydrate từ kho, fallback shard gốc

In [6]:
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T
import shutil

os.makedirs(LOCAL, exist_ok=True)
_n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
if _n < len(ORDER) * 0.999 and os.path.isdir(READY_X):
    for f in sorted(x for x in os.listdir(READY_X) if x.endswith('.tar')):
        src = os.path.join(READY_X, f); tmp = os.path.join('/content', f)
        try:
            shutil.copy(src, tmp)                        # copy TRUOC — khong mo tar tren mount
            assert os.path.getsize(tmp) == os.path.getsize(src), 'copy NGAN tu Drive'
            with tarfile.open(tmp) as tar: tar.extractall(LOCAL, filter='data')
        except Exception as e:
            print(f'  {f} HONG ({type(e).__name__}) -> bo qua')
        if os.path.exists(tmp): os.remove(tmp)
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
    print(f'hydrate tu kho: {_n:,} / {len(ORDER):,}')
if _n < len(ORDER) * 0.999:
    SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')
    assert os.path.isdir(SHARD_DIR), f'khong thay CA {READY_X} lan {SHARD_DIR}'
    TARS = sorted(os.path.join(SHARD_DIR, f) for f in os.listdir(SHARD_DIR)
                  if f.endswith('.tar') and f.startswith('lb_'))
    DONE = os.path.join(LOCAL, '_done.json')
    done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
    def _one(pl):
        nm, blob = pl; dst = os.path.join(LOCAL, os.path.basename(nm))
        if os.path.exists(dst): return
        try:
            im = Image.open(_io.BytesIO(blob)).convert('L').resize((SIZE, SIZE), Image.BICUBIC)
        except Exception:
            return                                       # mot anh hong KHONG giet ca lan chay
        tmp = dst + '.tmp'; im.save(tmp, 'JPEG', quality=95); os.replace(tmp, dst)   # NGUYEN TU
    with ThreadPoolExecutor(8) as pool:
        for tp in tqdm([t for t in TARS if os.path.basename(t) not in done], desc='stage', unit='tar'):
            with open(tp, 'rb') as f: buf = f.read()     # doc BULK; lap tren mount FUSE se TREO
            assert len(buf) == os.path.getsize(tp), f'{os.path.basename(tp)}: doc NGAN tren mount'
            with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
                jobs = [(m.name, tar.extractfile(m).read())
                        for m in tar.getmembers() if m.name.endswith('.jpg')]
            list(pool.map(_one, jobs)); del buf, jobs
            done.add(os.path.basename(tp)); json.dump(sorted(done), open(DONE, 'w'))
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
assert _n >= len(ORDER) * 0.999, f'thieu {len(ORDER)-_n:,} anh'
print(f'anh local: {_n:,} / {len(ORDER):,}')

proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MEAN, STD = proc.image_mean[0], proc.image_std[0]
eval_tf = T.Compose([T.ToTensor(), T.Normalize([MEAN]*3, [STD]*3)])

BB_SRC = os.path.join(OUT_DIR, 'stage4_nb38a', 'a2_E3', 'backbone_ft.pt')
assert os.path.exists(BB_SRC), f'khong thay {BB_SRC} -> chay 38a truoc'
_ck = torch.load(BB_SRC, map_location='cpu', weights_only=False)
backbone = AutoModel.from_pretrained('microsoft/rad-dino')
backbone.load_state_dict(_ck['bb'], strict=True)
backbone = backbone.to(dev).eval()
for p in backbone.parameters(): p.requires_grad_(False)
D_CXR = backbone.config.hidden_size
print(f'BACKBONE `ft` 38a/a2_E3 | d={D_CXR}')

class DS(Dataset):
    '''Tra ve CA HAI anh: hien tai va PHIM TRUOC (+ `has`).'''
    def __init__(s, idx, tf): s.idx, s.tf = np.asarray(idx), tf
    def __len__(s): return len(s.idx)
    def _img(s, i):
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        return s.tf(Image.merge('RGB', (im, im, im)))
    def __getitem__(s, k):
        i = int(s.idx[k]); pr = int(PREV_ROW[i]); x = s._img(i)
        return dict(image=x, image_prev=(s._img(pr) if pr >= 0 else torch.zeros_like(x)),
                    has=torch.tensor(1.0 if pr >= 0 else 0.0), row=i)

def loader(idx, tf, bs, nw=NW):
    return DataLoader(DS(idx, tf), batch_size=bs, shuffle=False, num_workers=nw,
                      pin_memory=True, persistent_workers=nw > 0,
                      prefetch_factor=4 if nw > 0 else None)
print('loader san sang')

hydrate tu kho: 4,000 / 377,110


stage:   0%|          | 0/378 [00:00<?, ?tar/s]

anh local: 377,110 / 377,110


preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

BACKBONE `ft` 38a/a2_E3 | d=768
loader san sang


## C3 · Kiến trúc — verbatim `53` r3 (có LayerScale), và `build_model()`

Chỉ dựng **một** arm (`t1_prior_both`). Việc **nạp** checkpoint nằm ở C4 — tách ra để dry-run chèn
được checkpoint giả giữa hai bước.

In [7]:
class GroupFC(nn.Module):
    def __init__(s, C, d):
        super().__init__()
        s.w = nn.Parameter(torch.randn(C, d) * d ** -0.5); s.b = nn.Parameter(torch.zeros(C))
    def forward(s, h): return (h * s.w).sum(-1) + s.b

class XAttn(nn.Module):
    def __init__(s, d, nhead=8, ff=2048, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, d))
    def forward(s, q, kv, need_w=False):
        a, w = s.at(q, kv, kv, need_weights=need_w, average_attn_weights=True)
        h = s.n1(q + a); return s.n2(h + s.ff(h)), w

class MicroBag(nn.Module):
    def __init__(s, V, d, const=False):
        super().__init__()
        s.const = const; s.w = nn.Parameter(torch.randn(V, d) * d ** -0.5)
        s.mlp = nn.Sequential(nn.GELU(), nn.Linear(d, d))
    def forward(s, ids, w):
        if s.const:
            ids = torch.zeros(ids.shape[0], 1, dtype=torch.long, device=ids.device)
            w = torch.ones(ids.shape[0], 1, device=ids.device)
        return s.mlp(F.embedding_bag(ids, s.w, per_sample_weights=w, mode='sum'))

class ColTokBag(nn.Module):
    def __init__(s, d_in, d=256, const=False, V=None):
        super().__init__()
        s.W = nn.Parameter(torch.randn(d_in, d) * d ** -0.5)
        s.b = nn.Parameter(torch.zeros(d_in, d))
        s.joint = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Linear(d, d))
        s.ln = nn.LayerNorm(d); s.const = const
        if const: s.c_in = nn.Parameter(torch.zeros(d_in))
        s.mb = MicroBag(V, d, const) if V else None
        if s.mb is not None:
            s.bln = nn.LayerNorm(d); s.cln = nn.LayerNorm(d) if const else nn.Identity()
    def forward(s, e, mids=None, mcnt=None):
        if s.const: e = s.c_in.unsqueeze(0).expand(e.shape[0], -1)
        col = s.b.unsqueeze(0) + e.unsqueeze(-1) * s.W.unsqueeze(0)
        base = s.ln(torch.cat([col, s.joint(e).unsqueeze(1)], 1))
        if s.mb is None: return base
        return torch.cat([s.cln(base), s.bln(s.mb(mids, mcnt)).unsqueeze(1)], 1)

class Tier1(nn.Module):
    def __init__(s, d, nhead=8, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d); s.pr = nn.Linear(d, d)
        nn.init.zeros_(s.pr.weight); nn.init.zeros_(s.pr.bias)
    def forward(s, patch, etok):
        a, _ = s.at(s.n(patch), etok, etok, need_weights=False)
        return patch + s.pr(s.ln_a(a))

class Tier1Img(nn.Module):
    '''Ban r3: LayerScale (`gamma` 1e-4) thay zero-init — khoa state_dict PHAI khop checkpoint.'''
    def __init__(s, d, nhead=8, drop=0.1, diff=True):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.nq = nn.LayerNorm(d); s.nk = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d)
        s.pr = nn.Linear(d, d)
        s.gamma = nn.Parameter(torch.full((d,), 1e-4))
        s.diff = diff
    def forward(s, patch, prior, has):
        a, _ = s.at(s.nq(patch), s.nk(prior), s.nk(prior), need_weights=False)
        z = s.gamma * s.pr(s.ln_a(patch - a if s.diff else a))
        return patch + z * has.view(-1, 1, 1)

class MLDecoder(nn.Module):
    def __init__(s, C, d_in, d=512, nhead=8, ff=1024, drop=0.1, d_ehr_in=0, t1=False):
        super().__init__()
        s.q = nn.Embedding(C, d); s.q.weight.requires_grad_(False)
        s.pin = nn.Linear(d_in, d); s.img = XAttn(d, nhead, ff, drop)
        s.t1 = Tier1(d, nhead, drop) if t1 else None
        if t1: s.ein = nn.Linear(d_ehr_in, d)
        s.out = GroupFC(C, d)
    def forward(s, patch, etok=None, need_w=False):
        q = s.q.weight.unsqueeze(0).expand(patch.shape[0], -1, -1)
        kv = s.pin(patch)
        if s.t1 is not None: kv = s.t1(kv, s.ein(etok))
        h, w = s.img(q, kv, need_w); return s.out(h), w

class ASL(nn.Module):
    '''Verbatim nb11..57. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()

def build_model():
    '''Mot arm `t1_prior_both` — dung de NAP checkpoint, khong bao gio huan luyen.'''
    return dict(mod=MLDecoder(C, D_CXR, D_MLD, d_ehr_in=D_ETOK, t1=True).to(dev),
                etk=ColTokBag(FEAT.shape[1], D_ETOK, False, V=V_MIC).to(dev),
                img=Tier1Img(D_CXR).to(dev))

from sklearn.metrics import average_precision_score
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
mac = lambda Y_, P_: float(np.nanmean(ap_cols(Y_, P_)))
tie = lambda Y_, P_: {t: float(np.nanmean(ap_cols(Y_, P_)[i])) for t, i in TIER.items()}
_r6 = np.random.default_rng(6); _y6 = (_r6.random((3000, C)) < 0.2).astype('float32')
_s6 = _r6.random((3000, C)).astype('float32')
assert np.nanmax(np.abs(ap_cols(_y6, _s6) - np.array(
    [average_precision_score(_y6[:, j], _s6[:, j]) for j in range(C)]))) < 1e-6
print('cong 1: ap_cols khop sklearn | build_model() san sang')

cong 1: ap_cols khop sklearn | build_model() san sang


## C4 · Cache `H` + **cổng tái lập** — cổng quan trọng nhất notebook

Cache sai (lệch hàng, sai thứ tự, quên phim trước, nhầm chế độ eval) thì **mọi** số phía sau vô
nghĩa mà không báo lỗi. Cổng: cho `H` đã cache qua đúng `GroupFC` của checkpoint phải **tái lập**
`p_{arm}_s{seed}_ep10.npy` mà `53` đã lưu — cùng mô hình, cùng ảnh, cùng transform, chỉ khác đường
đi qua cache fp16.

In [8]:
H_PATH = {s: os.path.join(CACHE_DIR, f'h59_s{s}.npy') for s in SEEDS}
W_CKPT = {}
def load_seed(seed):
    '''Nap DUNG mot arm tu checkpoint ModuleDict 7 arm cua 53 (khoa co tien to `t1_prior_both.`).'''
    _r = torch.load(os.path.join(SRC53, 'ckpt', f's{seed}.pt'), map_location=dev,
                    weights_only=False)
    M = build_model()
    for part in ('mod', 'etk', 'img'):
        sd = {k.split('.', 1)[1]: v for k, v in _r[part].items()
              if k.startswith(ARM53 + '.')}
        assert sd, (f'checkpoint s{seed} khong co khoa `{ARM53}.*` trong `{part}` — '
                    f'cac arm co: {sorted(set(k.split(".", 1)[0] for k in _r[part]))}')
        M[part].load_state_dict(sd, strict=True)         # strict: lech kien truc la chet NGAY o day
    for m in M.values():
        m.eval()
        for p in m.parameters(): p.requires_grad_(False)
    return M

MIDS_T, MCNT_T = torch.from_numpy(MIDS), torch.from_numpy(MCNT)
FEAT_T = torch.from_numpy(FEAT)

@torch.no_grad()
def cache_seed(seed):
    M = load_seed(seed)
    prog = H_PATH[seed] + '.prog'
    n0 = json.load(open(prog))['n'] if os.path.exists(prog) else 0
    FM = np.lib.format.open_memmap(H_PATH[seed], mode=('r+' if os.path.exists(H_PATH[seed])
                                                       else 'w+'),
                                   dtype='float16', shape=(len(ORDER), C, D_MLD))
    if n0 >= len(ORDER): print(f's{seed}: cache da du'); return
    rows = np.arange(n0, len(ORDER))
    t0 = time.time()
    for b in tqdm(loader(rows, eval_tf, BATCH), desc=f'cache s{seed}', leave=False):
        with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
            hl  = backbone(pixel_values=b['image'].to(dev, non_blocking=True)
                           ).last_hidden_state[:, 1:]
            hlp = backbone(pixel_values=b['image_prev'].to(dev, non_blocking=True)
                           ).last_hidden_state[:, 1:]
        hl, hlp = hl.float(), hlp.float()
        p  = M['img'](hl, hlp, b['has'].to(dev))
        er = b['row']
        et = M['etk'](FEAT_T[er].to(dev), MIDS_T[er].to(dev), MCNT_T[er].to(dev))
        kv = M['mod'].pin(p); kv = M['mod'].t1(kv, M['mod'].ein(et))
        q  = M['mod'].q.weight.unsqueeze(0).expand(kv.shape[0], -1, -1)
        h, _ = M['mod'].img(q, kv)
        FM[b['row'].numpy()] = h.half().cpu().numpy()
        n0 = int(b['row'].max()) + 1
        if n0 % (BATCH * 200) < BATCH:
            FM.flush(); json.dump({'n': n0}, open(prog, 'w'))
    FM.flush(); json.dump({'n': len(ORDER)}, open(prog, 'w'))
    W_CKPT[seed] = (M['mod'].out.w.detach().clone(), M['mod'].out.b.detach().clone())
    print(f's{seed}: cache xong {(time.time()-t0)/60:.1f} ph | {os.path.getsize(H_PATH[seed])/1e9:.1f} GB')
    del M; torch.cuda.empty_cache()

def gfc_logits(seed, rows, w=None, b=None, bs=65536):
    '''logit = (H * w).sum(-1) + b, theo lo tren GPU. Mac dinh dung GroupFC cua checkpoint.'''
    if w is None: w, b = W_CKPT[seed]
    FM = np.load(H_PATH[seed], mmap_mode='r')
    out = np.zeros((len(rows), C), 'float32')
    for i in range(0, len(rows), bs):
        hh = torch.from_numpy(np.asarray(FM[rows[i:i+bs]], 'float32')).to(dev)
        out[i:i+bs] = ((hh * w).sum(-1) + b).cpu().numpy()
    return out

sig = lambda x: 1.0 / (1.0 + np.exp(-x))
for s in SEEDS:
    cache_seed(s)
    if s not in W_CKPT:                                   # cache co san tu truoc -> van can W
        M = load_seed(s); W_CKPT[s] = (M['mod'].out.w.detach().clone(),
                                       M['mod'].out.b.detach().clone())
        del M; torch.cuda.empty_cache()
    # --- CONG TAI LAP: H cache + GroupFC checkpoint phai bang du doan 53 da luu ---
    PV = sig(gfc_logits(s, I_VA))
    REF = np.load(os.path.join(SRC53, f'p_{ARM53}_s{s}_ep{EP_REF}.npy'))
    _corr = float(np.corrcoef(PV.ravel(), REF.ravel())[0, 1])
    _dmap = abs(mac(YM[VA], PV) - mac(YM[VA], REF))
    print(f's{s} TAI LAP: corr {_corr:.6f} | lech mAP {_dmap:.5f}')
    assert _corr > 0.999 and _dmap < 1e-3, (
        f's{s}: cache KHONG tai lap duoc du doan cua 53 (corr {_corr:.4f}, lech mAP {_dmap:.4f}).\n'
        '  Cache dang LECH HANG hoac sai duong di forward -> moi so phia sau vo nghia. DUNG.')
print('cong 2: cache tai lap du doan nguon o CA CAC seed OK')

cache s43:   0%|          | 0/11785 [00:00<?, ?it/s]

s43: cache xong 28.0 ph | 15.4 GB
s43 TAI LAP: corr 1.000000 | lech mAP 0.00001
cong 2: cache tai lap du doan nguon o CA CAC seed OK


## C5 · Retrain `GroupFC` — 3 arm × seeds, trên đặc trưng đã cache

`cRT_inst` và `cRT_bal` **giống hệt nhau** (warm-start từ GroupFC checkpoint, cùng E, cùng LR, cùng
seed RNG) trừ đúng **phân bố lấy mẫu**. Trọng số mẫu cho bài đa nhãn:

```
w_i = sum_{j: y_ij=1} 1/n_j      n_j = so ca duong cua nhan j trong tap khop
w_i(anh toan am) = quantile 10% cua w duong    (san — tin hieu am khong bien mat)
```

In [9]:
crit = ASL(gn=4, gp=0).to(dev)
Y_TF_T = torch.from_numpy(YM[TF_]).to(dev)

# --- trong so lay mau can bang (chi tren tap KHOP) ---
n_pos = YM[TF_].sum(0)                                   # so ca duong moi nhan
w_bal = (YM[TF_] / np.maximum(n_pos, 1)[None, :]).sum(1)
_floor = np.quantile(w_bal[w_bal > 0], 0.10)
w_bal = np.maximum(w_bal, _floor).astype('float64'); w_bal /= w_bal.sum()

# cong 3: lay mau can bang phai DOI phan bo that su — nhan hiem nhat phai xuat hien day hon
_jr = int(np.argmin(np.where(n_pos > 0, n_pos, np.inf)))
_nat = float(YM[TF_][:, _jr].mean())
_bal_share = float((YM[TF_][:, _jr] * w_bal).sum())
print(f'nhan hiem nhat `{LAB40[_jr]}`: {int(n_pos[_jr])} ca duong | '
      f'tan suat tu nhien {_nat:.5f} -> can bang {_bal_share:.5f} ({_bal_share/max(_nat,1e-9):.1f}x)')
# Cong dat tren SAN TOAN HOC, khong tren ti le voi tu nhien: sampler can bang bao dam moi
# nhan nhan it nhat ~1/C khoi luong lay mau (tong w = so nhan co ca duong). Trong so hong
# (vd dong nhat) thi share = tan suat tu nhien 0.00059 << 0.02 -> bat. Ti le "x lan" chi de doc.
assert _bal_share >= 0.8 / C, (
    f'sampler can bang cho nhan hiem nhat chi {_bal_share:.5f} < {0.8/C:.4f} (~1/C) -> '
    'trong so lay mau sai hoac nhan hiem khong con trong tap khop')

def retrain(seed, balanced):
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed)            # CUNG seed hai arm -> ghep cap
    w0, b0 = W_CKPT[seed]
    w = nn.Parameter(w0.clone()); b = nn.Parameter(b0.clone())   # WARM START tu checkpoint
    opt = torch.optim.AdamW([{'params': [w], 'weight_decay': WD_RT},
                             {'params': [b], 'weight_decay': 0.0}], lr=LR_RT)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, E_RT)
    FM = np.load(H_PATH[seed], mmap_mode='r')
    H_TF = torch.from_numpy(np.asarray(FM[I_TF]))        # fp16 vao RAM (~9 GB) — doc mot lan
    N = len(I_TF); pv_eps, ps_eps, pt_eps = [], [], []
    wb = torch.from_numpy(w_bal)
    for ep in range(E_RT):
        idx = (torch.multinomial(wb, N, replacement=True, generator=gen) if balanced
               else torch.randperm(N, generator=gen))
        for i in range(0, N, BS_RT):
            k = idx[i:i+BS_RT]
            hh = H_TF[k].to(dev, non_blocking=True).float()
            lg = (hh * w).sum(-1) + b
            opt.zero_grad(set_to_none=True); crit(lg, Y_TF_T[k]).backward(); opt.step()
        sch.step()
        wd_, bd_ = w.detach(), b.detach()
        pv_eps.append(sig(gfc_logits(seed, I_VA, wd_, bd_)))
        ps_eps.append(sig(gfc_logits(seed, I_SL, wd_, bd_)))
        if DO_TEST: pt_eps.append(sig(gfc_logits(seed, I_TE, wd_, bd_)).astype('float16'))
    del H_TF
    return pv_eps, ps_eps, pt_eps

RT = {}
for s in SEEDS:
    for tag, bal in (('cRT_inst', False), ('cRT_bal', True)):
        t0 = time.time()
        RT[(s, tag)] = retrain(s, bal)
        pv = np.mean(RT[(s, tag)][0], 0)                 # ensemble TAT CA epoch retrain (giao thuc 58)
        print(f's{s} {tag:<9} {len(RT[(s, tag)][0])} ep | {(time.time()-t0)/60:.1f} ph | '
              f'val mAP (ens all) {mac(YM[VA], pv):.4f} | tail {tie(YM[VA], pv)["tail"]:.4f}')
        ensure_res()
        np.save(os.path.join(RES59, f'pens_{tag}_s{s}.npy'), pv.astype('float16'))

nhan hiem nhat `Lobar Atelectasis`: 109 ca duong | tan suat tu nhien 0.00050 -> can bang 0.02538 (51.2x)
s43 cRT_inst  10 ep | 1.6 ph | val mAP (ens all) 0.2482 | tail 0.0954
s43 cRT_bal   10 ep | 1.3 ph | val mAP (ens all) 0.2475 | tail 0.0970


## C6 · Kết quả + bootstrap ghép cặp + cổng

Mọi arm xử lý **y hệt**: ensemble tất cả epoch retrain × 2 seed. `base` = GroupFC checkpoint (một
"epoch", nhưng ghép cặp `bal − inst` không đụng tới nó — nó chỉ là mốc ngữ cảnh).

In [10]:
ENS = {}
for tag in ('cRT_inst', 'cRT_bal'):
    ENS[tag] = np.mean([np.mean(RT[(s, tag)][0], 0) for s in SEEDS], 0)
ENS['base'] = np.mean([sig(gfc_logits(s, I_VA)) for s in SEEDS], 0)

print(f'{"arm":<12}{"macro":>9}{"head":>9}{"medium":>9}{"tail":>9}')
for tag in ('base', 'cRT_inst', 'cRT_bal'):
    t = tie(YM[VA], ENS[tag])
    print(f'  {tag:<10}{mac(YM[VA], ENS[tag]):>9.4f}{t["head"]:>9.4f}{t["medium"]:>9.4f}'
          f'{t["tail"]:>9.4f}')
G_tr = mac(YM[VA], ENS['cRT_inst']) - mac(YM[VA], ENS['base'])
G_bal = mac(YM[VA], ENS['cRT_bal']) - mac(YM[VA], ENS['cRT_inst'])
G_bal_t = tie(YM[VA], ENS['cRT_bal'])['tail'] - tie(YM[VA], ENS['cRT_inst'])['tail']
print(f'\n  cRT_inst - base   {G_tr:+.4f}   (phan CHI do huan luyen them)')
print(f'  G_bal  macro      {G_bal:+.4f}')
print(f'  G_bal  tail       {G_bal_t:+.4f}   <- CAU HOI CHINH')

# --- bootstrap GHEP CAP theo BENH NHAN ---
B = 1000
SUB_VA = SUB[VA]; UQ = np.unique(SUB_VA)
IDX_OF = {u: np.where(SUB_VA == u)[0] for u in UQ}
def pboot(pa, pb, cols, seed=4):
    r = np.random.default_rng(seed); d = []
    for _ in range(B):
        ix = np.concatenate([IDX_OF[u] for u in r.choice(UQ, size=len(UQ), replace=True)])
        aa, bb = ap_cols(YM[VA][ix], pa[ix]), ap_cols(YM[VA][ix], pb[ix])
        if cols is not None: aa, bb = aa[cols], bb[cols]
        d.append(float(np.nanmean(bb)) - float(np.nanmean(aa)))
    d = np.array(d); return float(d.mean()), *np.percentile(d, [2.5, 97.5])
for nm, cols in (('macro', None), ('tail', TIER['tail'])):
    m, lo, hi = pboot(ENS['cRT_inst'], ENS['cRT_bal'], cols)
    ok = 'LOAI 0' if lo * hi > 0 else 'chua 0'
    print(f'  G_bal {nm:>6}  {m:+.4f}  CI95 [{lo:+.4f}, {hi:+.4f}]  {ok}')

print('\n' + '=' * 70)
print('CONG KHOA TRUOC:')
print('  NHAN      G_bal tail > +0.005  VA  CI loai 0')
print('  KHONG     macro tut qua -0.002')
print('  VUNG XAM  CI chua 0 nhung diem uoc luong > +0.003 -> THEM SEED')
_m, _lo, _hi = pboot(ENS['cRT_inst'], ENS['cRT_bal'], TIER['tail'], seed=5)
_v = ('NHAN' if (_m > 0.005 and _lo * _hi > 0 and G_bal > -0.002) else
      'VUNG XAM (them seed)' if _m > 0.003 else 'DONG: classifier khong phai nut that -> nb55')
print(f'  -> {_v}')
print('=' * 70)

if DO_TEST:
    PT = {tag: np.mean([np.mean(RT[(s, tag)][2], 0).astype('float32') for s in SEEDS], 0)
          for tag in ('cRT_inst', 'cRT_bal')}
    with open(P_TOUCH, 'a') as _f:
        _f.write(json.dumps(dict(nb='59', run=RUN_TAG, n=len(I_TE), arms=list(PT),
                                 muc_dich='theo doi',
                                 t=datetime.datetime.now().isoformat(timespec='seconds'))) + '\n')
    print('\n=== TEST (CHI theo doi — khong quyet dinh gi tren so nay) ===')
    for tag, p in PT.items():
        t = tie(YM[TE], p)
        print(f'  {tag:<10} macro {mac(YM[TE], p):.4f} | tail {t["tail"]:.4f}')
    print('  !! model chi thay 74% du lieu co nhan -> CAN DUOI, khong phai con so cuoi.')

L = [f'# REPORT · `nb59` — decoupled classifier (cRT), nguon `{SRC53}` arm `{ARM53}`', '',
     f'Giao thuc: ensemble {E_RT} epoch retrain x {len(SEEDS)} seed, ghep cap, '
     'bootstrap theo benh nhan.', '',
     '| arm | macro | tail |', '|---|---|---|']
for tag in ('base', 'cRT_inst', 'cRT_bal'):
    L.append(f'| `{tag}` | {mac(YM[VA], ENS[tag]):.4f} | {tie(YM[VA], ENS[tag])["tail"]:.4f} |')
L += ['', f'`G_bal tail = {G_bal_t:+.4f}` | `G_bal macro = {G_bal:+.4f}` | '
      f'`cRT_inst - base = {G_tr:+.4f}`', '', f'Ket luan cong: **{_v}**', '',
      '## Nhung gi ket qua nay KHONG tra loi', '',
      '- Chi mo `GroupFC`. Mo them `XAttn` (dinh tuyen theo lop) la bac dat hon, chi leo khi bac '
      'nay duong.', '- Khong dung den bieu dien: neu tail bi chan boi bieu dien thi G_bal ~ 0 va '
      'cau tra loi la `nb55`.']
ensure_res()
open(os.path.join(RES59, 'REPORT_STAGE4_NB59.md'), 'w', encoding='utf-8').write('\n'.join(L))
print('\nda ghi ' + os.path.join(RES59, 'REPORT_STAGE4_NB59.md'))

arm             macro     head   medium     tail
  base         0.2492   0.4647   0.1792   0.0964
  cRT_inst     0.2482   0.4642   0.1777   0.0954
  cRT_bal      0.2475   0.4627   0.1762   0.0970

  cRT_inst - base   -0.0010   (phan CHI do huan luyen them)
  G_bal  macro      -0.0006
  G_bal  tail       +0.0015   <- CAU HOI CHINH
  G_bal  macro  -0.0006  CI95 [-0.0021, +0.0011]  chua 0
  G_bal   tail  +0.0016  CI95 [-0.0029, +0.0069]  chua 0

CONG KHOA TRUOC:
  NHAN      G_bal tail > +0.005  VA  CI loai 0
  KHONG     macro tut qua -0.002
  VUNG XAM  CI chua 0 nhung diem uoc luong > +0.003 -> THEM SEED
  -> DONG: classifier khong phai nut that -> nb55

=== TEST (CHI theo doi — khong quyet dinh gi tren so nay) ===
  cRT_inst   macro 0.2409 | tail 0.0812
  cRT_bal    macro 0.2393 | tail 0.0819
  !! model chi thay 74% du lieu co nhan -> CAN DUOI, khong phai con so cuoi.

da ghi /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/stage4_nb59/r1/REPORT_STAGE4_NB59.md


## C7 · Quét `LR_RT` — kiểm `G_bal` có bị đo trong chế độ đã hỏng không

`cRT_inst` là huấn luyện lại với **đúng phân bố tự nhiên**, nên nó phải xấp xỉ `base`. Ở `LR=1e-3`
nó lại **thua** `-0.0010` — dù còn được lợi thế miễn phí là ensemble 10 epoch retrain trong khi
`base` chỉ là một điểm. Đó là dấu hiệu LR quá cao: nó đẩy classifier ra khỏi nghiệm tốt sẵn có.

`G_bal` vẫn hợp lệ (hai arm cùng LR, ghép cặp), nhưng đang đo **trong chế độ đã bị làm hỏng** — một
hiệu ứng nhỏ rất dễ chìm ở đó.

**Tiêu chí lành mạnh, khoá trước**: chọn `LR` mà `|cRT_inst − base| < 0.0005`. Chỉ khi việc huấn
luyện lại **không phá** gì thì `G_bal` mới đo đúng thứ nó định đo. Rồi đọc lại `G_bal` ở `LR` đó.

Cache đã nằm trên đĩa nên mỗi `LR` chỉ tốn ~3 phút.

In [ ]:
LR_GRID = (1e-4, 3e-4, 1e-3)
_base_m = {s: mac(YM[VA], sig(gfc_logits(s, I_VA))) for s in SEEDS}
BASE_M = float(np.mean(list(_base_m.values())))
BASE_T = tie(YM[VA], np.mean([sig(gfc_logits(s, I_VA)) for s in SEEDS], 0))['tail']
print(f'base  macro {BASE_M:.4f} | tail {BASE_T:.4f}\n')
print(f'{"LR":>8}{"inst":>9}{"bal":>9}{"|inst-base|":>13}{"G_bal mac":>11}{"G_bal tail":>12}  lanh manh')
SWEEP = {}
_LR0 = LR_RT
for _lr in LR_GRID:
    LR_RT = _lr                                          # `retrain()` doc bien toan cuc nay
    E = {}
    for tag, bal in (('cRT_inst', False), ('cRT_bal', True)):
        E[tag] = np.mean([np.mean(retrain(s, bal)[0], 0) for s in SEEDS], 0)
    d_inst = abs(mac(YM[VA], E['cRT_inst']) - BASE_M)
    gm = mac(YM[VA], E['cRT_bal']) - mac(YM[VA], E['cRT_inst'])
    gt = tie(YM[VA], E['cRT_bal'])['tail'] - tie(YM[VA], E['cRT_inst'])['tail']
    SWEEP[_lr] = dict(E=E, d_inst=d_inst, gm=gm, gt=gt)
    print(f'{_lr:>8.0e}{mac(YM[VA], E["cRT_inst"]):>9.4f}{mac(YM[VA], E["cRT_bal"]):>9.4f}'
          f'{d_inst:>13.4f}{gm:>+11.4f}{gt:>+12.4f}  {"OK" if d_inst < 5e-4 else "PHA"}')
LR_RT = _LR0

_ok = [lr for lr in LR_GRID if SWEEP[lr]['d_inst'] < 5e-4]
print('\n' + '=' * 70)
if not _ok:
    _best = min(LR_GRID, key=lambda lr: SWEEP[lr]['d_inst'])
    print(f'KHONG LR nao dat tieu chi lanh manh (|inst-base| < 0.0005).')
    print(f'  Gan nhat: LR={_best:.0e} lech {SWEEP[_best]["d_inst"]:.4f}.')
    print('  Nghia la MOI lan huan luyen lai deu lam xau di => classifier DA HOI TU,')
    print('  va gia thuyet goc cua huong #2 ("classifier tail chua hoc xong") SAI.')
else:
    _best = max(_ok, key=lambda lr: SWEEP[lr]['gt'])     # trong cac LR lanh manh, lay G_bal tail cao nhat
    S = SWEEP[_best]
    print(f'LR lanh manh: {[f"{lr:.0e}" for lr in _ok]} -> chon {_best:.0e}')
    m, lo, hi = pboot(S['E']['cRT_inst'], S['E']['cRT_bal'], TIER['tail'], seed=6)
    print(f'  G_bal tail {m:+.4f}  CI95 [{lo:+.4f}, {hi:+.4f}]  '
          f'{"LOAI 0" if lo*hi > 0 else "chua 0"}   | G_bal macro {S["gm"]:+.4f}')
    _v2 = ('NHAN' if (m > 0.005 and lo*hi > 0 and S['gm'] > -0.002) else
           'VUNG XAM (them seed)' if m > 0.003 else
           'DONG: classifier khong phai nut that -> nb55')
    print(f'  -> {_v2}')
print('=' * 70)
print('Bat ke ket qua: `cRT_inst` KHONG vuot duoc `base` du co them 10 epoch VA ensemble')
print('la bang chung doc lap rang classifier tuyen tinh tren dac trung dong cung DA HOI TU.')
ensure_res()
pd.DataFrame([dict(lr=lr, d_inst=v['d_inst'], g_macro=v['gm'], g_tail=v['gt'])
              for lr, v in SWEEP.items()]).to_csv(os.path.join(RES59, 'lr_sweep.csv'), index=False)